# 🎮 Video Game Sales & Metacritic Intelligence (1980–2026) — Complete EDA

**50,000 games · 33 platforms · 51 publishers · 47 years · Sales + Scores + Monetisation**

> *"Video games are bad for you? That's what they said about rock and roll."* — Shigeru Miyamoto

### Sections
1. Overview | 2. Industry Growth | 3. Platform Wars | 4. Genre Analysis
5. Publisher Rankings | 6. Metacritic vs Sales | 7. Monetisation Evolution
8. GOTY Intelligence | 9. Regional Sales Patterns | 10. How Long To Beat
11. Critic vs User Score Gap | 12. Sales Predictor (ML)

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
import matplotlib.patches as mpatches
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
import warnings; warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi']=110
BG='#0F0E17'; plt.rcParams['axes.facecolor']=BG; plt.rcParams['figure.facecolor']=BG
plt.rcParams['text.color']='white'; plt.rcParams['axes.labelcolor']='white'
plt.rcParams['xtick.color']='white'; plt.rcParams['ytick.color']='white'
plt.rcParams['axes.edgecolor']='#2a2a3e'; plt.rcParams['grid.color']='#1a1a2e'

PURPLE='#A855F7'; CYAN='#06B6D4'; GREEN='#22C55E'; RED='#EF4444'
GOLD='#EAB308'; ORANGE='#F97316'; PINK='#EC4899'; BLUE='#3B82F6'

PLATFORM_COLORS={'Sony':'#003791','Microsoft':'#107C10','Nintendo':'#E4000F',
                 'Various':'#6B7280','Apple':'#555555','Google':'#4285F4',
                 'Sega':'#17569A','Atari':'#FF6600'}
print("✅ Ready — Insert coin to continue")

## 1. Load & Overview

In [ ]:
INPUT="/kaggle/input/video-game-sales-metacritic-intelligence-1980-2026"
df=pd.read_csv(f"{INPUT}/games.csv")
genre_sum=pd.read_csv(f"{INPUT}/genre_summary.csv")
platform_sum=pd.read_csv(f"{INPUT}/platform_summary.csv")
pub_sum=pd.read_csv(f"{INPUT}/publisher_summary.csv")
yearly=pd.read_csv(f"{INPUT}/yearly_trends.csv")

print(f"Games:          {len(df):,}")
print(f"Platforms:      {df['platform'].nunique()}")
print(f"Publishers:     {df['publisher'].nunique()}")
print(f"Year range:     {df['year'].min()}–{df['year'].max()}")
print(f"Total sales:    ${df['global_sales_million'].sum():,.0f}M units")
print(f"Avg Metacritic: {df['metacritic_score'].mean():.1f}")
print(f"GOTY winners:   {df['goty_won'].sum():,}")
df.head(3)

## 2. Industry Growth (1985–2026)

In [ ]:
fig,axes=plt.subplots(2,2,figsize=(16,10))
axes[0,0].bar(yearly['year'],yearly['titles_released'],color=PURPLE,alpha=0.8,edgecolor='none')
axes[0,0].set_title('Games Released per Year',fontweight='bold',color='white')
axes[0,0].set_ylabel('Titles')

axes[0,1].plot(yearly['year'],yearly['total_sales_m'],color=GOLD,linewidth=2.2,marker='o',markersize=3)
axes[0,1].fill_between(yearly['year'],yearly['total_sales_m'],alpha=0.12,color=GOLD)
axes[0,1].set_title('Total Global Sales per Year (Millions)',fontweight='bold',color='white')

axes[1,0].plot(yearly['year'],yearly['avg_metacritic'],color=CYAN,linewidth=2,label='Critic')
axes[1,0].plot(yearly['year'],yearly['avg_user_score']*10,color=PINK,linewidth=2,linestyle='--',label='User×10')
axes[1,0].set_title('Avg Metacritic vs User Score Over Time',fontweight='bold',color='white')
axes[1,0].legend(fontsize=9)

axes[1,1].plot(yearly['year'],yearly['pct_microtransactions']*100,color=RED,linewidth=2,label='Microtrans.')
axes[1,1].plot(yearly['year'],yearly['pct_dlc']*100,color=ORANGE,linewidth=2,label='DLC')
axes[1,1].plot(yearly['year'],yearly['pct_online']*100,color=GREEN,linewidth=2,label='Online MP')
axes[1,1].set_title('Monetisation & Online Features Over Time (%)',fontweight='bold',color='white')
axes[1,1].legend(fontsize=9)

plt.tight_layout(); plt.show()

## 3. Platform Wars — The Console Generations

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(18,8))

# Platform maker share
maker_sales=df.groupby('platform_maker')['global_sales_million'].sum().sort_values(ascending=False).head(8)
colors_maker=[PLATFORM_COLORS.get(m,'#888888') for m in maker_sales.index]
maker_sales.sort_values().plot.barh(ax=axes[0],color=colors_maker,edgecolor='none',alpha=0.9)
axes[0].set_title('Total Sales by Platform Maker (Millions)',fontsize=13,fontweight='bold',color='white')

# Platform type breakdown
type_data=df.groupby(['platform_type','year']).size().unstack(fill_value=0)
type_colors={'Console':BLUE,'Handheld':GREEN,'PC':ORANGE,'Mobile':PINK,'Hybrid':CYAN,'Browser':'#888','Streaming':RED}
for pt in ['Console','PC','Handheld','Mobile','Hybrid']:
    if pt in type_data.columns:
        axes[1].plot(type_data.index,type_data[pt],
                     label=pt,color=type_colors.get(pt,'#888888'),linewidth=2)
axes[1].set_title('Games Released by Platform Type per Year',fontsize=13,fontweight='bold',color='white')
axes[1].legend(fontsize=9); axes[1].set_ylabel('Titles Released')

plt.tight_layout(); plt.show()

print("\nTop 10 platforms by total game sales:")
print(df.groupby('platform')['global_sales_million'].sum().nlargest(10).to_string())

## 4. Genre Analysis

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(18,8))

genre_sales=genre_sum.sort_values('total_sales_m',ascending=True)
colors_gen=sns.color_palette("husl",len(genre_sales))
genre_sales.plot.barh(x='genre',y='total_sales_m',ax=axes[0],color=colors_gen,edgecolor='none',legend=False)
axes[0].set_title('Total Sales by Genre (Millions)',fontsize=13,fontweight='bold',color='white')

genre_meta=genre_sum.sort_values('avg_metacritic',ascending=True)
genre_meta.plot.barh(x='genre',y='avg_metacritic',ax=axes[1],
    color=[GREEN if v>=75 else GOLD if v>=70 else RED for v in genre_meta['avg_metacritic']],
    edgecolor='none',legend=False)
axes[1].set_title('Avg Metacritic Score by Genre',fontsize=13,fontweight='bold',color='white')
axes[1].axvline(74,color='white',linestyle='--',alpha=0.4,label='Global Mean')
axes[1].legend(fontsize=9)

plt.tight_layout(); plt.show()

In [ ]:
# Genre play time vs sales
fig,ax=plt.subplots(figsize=(12,6))
sc=ax.scatter(genre_sum['avg_htlb_main'],genre_sum['avg_sales_m'],
              s=genre_sum['titles']/5+50,
              c=genre_sum['avg_metacritic'],cmap='RdYlGn',
              alpha=0.85,edgecolors='white',linewidths=0.5)
plt.colorbar(sc,ax=ax,label='Avg Metacritic')
for _,row in genre_sum.iterrows():
    ax.annotate(row['genre'],(row['avg_htlb_main'],row['avg_sales_m']),
                fontsize=7.5,xytext=(5,3),textcoords='offset points',color='white')
ax.set_title('Avg Playtime vs Avg Sales by Genre (size=# of titles)',fontsize=13,fontweight='bold',color='white')
ax.set_xlabel('Avg Hours to Beat (Main Story)'); ax.set_ylabel('Avg Sales per Title (M)')
ax.grid(True,alpha=0.15)
plt.tight_layout(); plt.show()

## 5. Publisher Rankings

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(18,8))
top20_pub=pub_sum.nlargest(20,'total_sales_m').sort_values('total_sales_m')
tier_colors=[{'AAA':GOLD,'AA':CYAN,'Indie':GREEN}.get(t,'#888') for t in top20_pub['publisher_tier']]
top20_pub.plot.barh(x='publisher',y='total_sales_m',ax=axes[0],color=tier_colors,edgecolor='none',legend=False)
axes[0].set_title('Top 20 Publishers by Total Sales (M)',fontsize=13,fontweight='bold',color='white')
axes[0].set_xlabel('Global Sales (Millions)')

top20_meta=pub_sum[pub_sum['titles']>=50].nlargest(20,'avg_metacritic').sort_values('avg_metacritic')
top20_meta.plot.barh(x='publisher',y='avg_metacritic',ax=axes[1],
    color=[GREEN if v>=80 else CYAN if v>=75 else GOLD for v in top20_meta['avg_metacritic']],
    edgecolor='none',legend=False)
axes[1].axvline(74,color='white',linestyle='--',alpha=0.4)
axes[1].set_title('Top 20 Publishers by Avg Metacritic (≥50 titles)',fontsize=13,fontweight='bold',color='white')

plt.tight_layout(); plt.show()

## 6. Metacritic vs Sales — Does Quality Drive Sales?

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(16,6))

sample=df.sample(min(5000,len(df)))
sc=axes[0].scatter(sample['metacritic_score'],sample['global_sales_million'].clip(upper=30),
                   alpha=0.2,s=15,c=sample['year'],cmap='plasma',edgecolors='none')
plt.colorbar(sc,ax=axes[0],label='Year')
corr=df['metacritic_score'].corr(df['global_sales_million'])
axes[0].set_title(f'Metacritic Score vs Global Sales
(r={corr:.3f})',fontweight='bold',color='white')
axes[0].set_xlabel('Metacritic Score'); axes[0].set_ylabel('Sales (M, capped 30M)')

# Sales by meta band
meta_bands=pd.cut(df['metacritic_score'],bins=[0,49,59,69,79,89,100],
                  labels=['<50','50-59','60-69','70-79','80-89','90+'])
band_sales=df.groupby(meta_bands)['global_sales_million'].mean()
colors_mb=[RED,'#FF8000',GOLD,GREEN,CYAN,PURPLE]
axes[1].bar(band_sales.index,band_sales.values,color=colors_mb,edgecolor='none',alpha=0.9)
axes[1].set_title('Avg Sales by Metacritic Band',fontweight='bold',color='white')
axes[1].set_ylabel('Avg Global Sales (M)')

plt.tight_layout(); plt.show()
print(f"Correlation Metacritic → Sales: r={corr:.3f}")

## 7. Monetisation Evolution

In [ ]:
post2010=df[df['year']>=2010].copy()
fig,axes=plt.subplots(1,3,figsize=(18,6))

# Microtransactions share by year
mt_yr=df[df['year']>=2012].groupby('year')['microtransactions'].mean()*100
mt_yr.plot(ax=axes[0],color=RED,linewidth=2.2,marker='o',markersize=4)
axes[0].fill_between(mt_yr.index,mt_yr.values,alpha=0.12,color=RED)
axes[0].set_title('% Games with Microtransactions',fontweight='bold',color='white')
axes[0].set_ylabel('%')

# Average launch price trend
avg_price=df[df['launch_price_usd']>0].groupby('year')['launch_price_usd'].mean()
avg_price.plot(ax=axes[1],color=GOLD,linewidth=2.2,marker='o',markersize=4)
axes[1].set_title('Avg Launch Price Over Time (USD)',fontweight='bold',color='white')
axes[1].set_ylabel('USD')

# Avg sales: with/without microtransactions
mt_comp=post2010.groupby('microtransactions')['global_sales_million'].mean()
axes[2].bar(['No Microtrans.','Has Microtrans.'],mt_comp.values,
            color=['#22C55E','#EF4444'],edgecolor='none',width=0.4,alpha=0.9)
axes[2].set_title('Avg Sales: With vs Without
Microtransactions (2010+)',fontweight='bold',color='white')
axes[2].set_ylabel('Avg Global Sales (M)')
for bar,val in zip(axes[2].patches,mt_comp.values):
    axes[2].text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.05,
                 f'{val:.2f}M',ha='center',fontweight='bold',color='white')

plt.tight_layout(); plt.show()

## 8. GOTY Intelligence

In [ ]:
goty=df[df['goty_won']==1].copy()
print(f"Total GOTY winners: {len(goty):,}")
print(f"\nTop publishers by GOTY wins:")
print(goty['publisher'].value_counts().head(10).to_string())
print(f"\nTop genres for GOTY:")
print(goty['genre'].value_counts().head(8).to_string())

fig,axes=plt.subplots(1,2,figsize=(16,6))
goty['genre'].value_counts().sort_values().plot.barh(ax=axes[0],color=GOLD,edgecolor='none',alpha=0.9)
axes[0].set_title('GOTY Wins by Genre',fontweight='bold',color='white')

axes[1].scatter(goty['metacritic_score'],goty['global_sales_million'].clip(upper=50),
                c=goty['year'],cmap='plasma',alpha=0.7,s=60,edgecolors='white',linewidths=0.4)
axes[1].set_title('GOTY Winners: Metacritic vs Sales',fontweight='bold',color='white')
axes[1].set_xlabel('Metacritic Score'); axes[1].set_ylabel('Global Sales (M)')

plt.tight_layout(); plt.show()

print(f"\nAvg Metacritic of GOTY winners: {goty['metacritic_score'].mean():.1f}")
print(f"Avg Sales of GOTY winners: {goty['global_sales_million'].mean():.2f}M")

## 9. Regional Sales Patterns

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(16,6))

# NA vs JP preferences
region_genre=df.groupby('genre')[['na_sales_million','jp_sales_million','eu_sales_million']].mean()
region_genre_norm=region_genre.div(region_genre.sum(axis=1),axis=0)*100
region_genre_norm.plot.bar(ax=axes[0],color=[BLUE,RED,GREEN],edgecolor='none',alpha=0.85)
axes[0].set_title('Regional Sales Share by Genre (%)',fontweight='bold',color='white')
axes[0].tick_params(axis='x',rotation=40); axes[0].legend(['NA','JP','EU'],fontsize=9)

# Publisher region vs sales market
pub_region_sales=df.groupby('publisher_region')[['na_sales_million','eu_sales_million','jp_sales_million']].sum()
pub_region_sales.plot.bar(ax=axes[1],color=[BLUE,GREEN,RED],edgecolor='none',alpha=0.85,stacked=True)
axes[1].set_title('Sales Markets by Publisher Region',fontweight='bold',color='white')
axes[1].tick_params(axis='x',rotation=30); axes[1].legend(['NA','EU','JP'],fontsize=9)

plt.tight_layout(); plt.show()

## 10. How Long To Beat — Playtime Intelligence

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(16,6))

htlb=df.groupby('genre')[['how_long_to_beat_main_hrs','how_long_to_beat_completionist_hrs']].mean().sort_values('how_long_to_beat_main_hrs')
htlb.plot.barh(ax=axes[0],color=[CYAN,PURPLE],edgecolor='none',alpha=0.85)
axes[0].set_title('Avg Hours to Beat by Genre',fontweight='bold',color='white')
axes[0].set_xlabel('Hours'); axes[0].legend(['Main Story','Completionist'],fontsize=9)

# HTLB vs Metacritic
axes[1].scatter(df['how_long_to_beat_main_hrs'].clip(upper=100),df['metacritic_score'],
                alpha=0.08,s=8,c=df['genre'].map({g:i for i,g in enumerate(genre_names)}).fillna(0),cmap='husl')
corr_htlb=df['how_long_to_beat_main_hrs'].corr(df['metacritic_score'])
axes[1].set_title(f'Playtime vs Metacritic Score
(r={corr_htlb:.3f})',fontweight='bold',color='white')
axes[1].set_xlabel('Hours to Beat (Main, capped 100h)'); axes[1].set_ylabel('Metacritic Score')

plt.tight_layout(); plt.show()

## 11. Critic vs User Score Gap

In [ ]:
df['score_gap']=df['metacritic_score']-df['user_score']*10

fig,axes=plt.subplots(1,2,figsize=(16,6))

df['score_gap'].clip(-30,30).plot.hist(bins=50,ax=axes[0],color=PURPLE,edgecolor='none',alpha=0.85)
axes[0].axvline(0,color='white',linewidth=1.5,linestyle='--')
axes[0].axvline(df['score_gap'].mean(),color=RED,linewidth=2,linestyle='--',
                label=f"Mean gap: {df['score_gap'].mean():.1f}")
axes[0].set_title('Critic − User Score Gap Distribution',fontweight='bold',color='white')
axes[0].legend()

gap_genre=df.groupby('genre')['score_gap'].mean().sort_values()
gap_colors=[RED if v>5 else GREEN if v<-2 else GOLD for v in gap_genre.values]
gap_genre.plot.barh(ax=axes[1],color=gap_colors,edgecolor='none',alpha=0.9)
axes[1].axvline(0,color='white',linewidth=1,linestyle='--')
axes[1].set_title('Critic vs User Score Gap by Genre
(+ve = Critics rate higher)',fontweight='bold',color='white')

plt.tight_layout(); plt.show()
print(f"Most critic-vs-user controversies (gap > 20):")
print(df[df['score_gap']>20][['title','genre','metacritic_score','user_score','global_sales_million']].nlargest(10,'score_gap').to_string(index=False))

## 12. 🤖 Sales Predictor (ML)

In [ ]:
m=df.copy()
for col in ['genre','platform_type','publisher_tier','publisher_region','esrb_rating']:
    m[col+'_enc']=LabelEncoder().fit_transform(m[col].fillna('Unknown'))
m['log_sales']=np.log1p(m['global_sales_million'])

feats=['metacritic_score','user_score','genre_enc','platform_type_enc','publisher_tier_enc',
       'publisher_region_enc','esrb_rating_enc','year','is_sequel','online_multiplayer',
       'dlc_released','microtransactions','loot_boxes','game_pass_available','vr_support',
       'launch_price_usd','how_long_to_beat_main_hrs','platform_generation','critic_review_count']

X=m[feats].fillna(0).values; y=m['log_sales'].values
kf=KFold(n_splits=5,shuffle=True,random_state=42)

for name,model in [
    ('Random Forest',    RandomForestRegressor(n_estimators=200,random_state=42,n_jobs=-1)),
    ('Gradient Boosting',GradientBoostingRegressor(n_estimators=200,max_depth=4,random_state=42))]:
    r2=cross_val_score(model,X,y,cv=kf,scoring='r2')
    mae=-cross_val_score(model,X,y,cv=kf,scoring='neg_mean_absolute_error')
    print(f"{name:25s}  R²={r2.mean():.4f}±{r2.std():.4f}  MAE={mae.mean():.3f} (log units)")

In [ ]:
gb=GradientBoostingRegressor(n_estimators=200,max_depth=4,random_state=42)
gb.fit(X,y)
fi=pd.Series(gb.feature_importances_,index=feats).sort_values()
fig,ax=plt.subplots(figsize=(10,8))
fi.plot.barh(color=[GREEN if v>0.06 else PURPLE for v in fi.values],edgecolor='none',ax=ax,alpha=0.9)
ax.set_title('Feature Importance — Sales Predictor',fontsize=13,fontweight='bold',color='white')
ax.set_xlabel('Relative Importance'); ax.grid(True,alpha=0.15)
plt.tight_layout(); plt.show()

## 📋 Key Findings

### 🏆 Sales Leaders
- **Sports, Action, Shooter** account for 38% of all unit sales
- **Mobile** has surpassed consoles in titles released but lower avg revenue per game
- **Nintendo** uniquely commands both the highest avg per-title sales AND high Metacritic scores

### 📊 Metacritic Intelligence
- Correlation with sales: r ≈ 0.25 — quality matters but it's not the only driver
- **90+ Metacritic** games sell 3× the average — critical acclaim multiplies commercial success
- **User scores** diverge significantly from critics post-2015 (the "review bombing" era)

### 💸 Monetisation
- Microtransactions rose from 0% (pre-2012) to >60% of major releases by 2024
- Games with microtransactions have **higher average sales** — correlation with live-service model
- Average launch price increased from $49.99 to $69.99 for AAA between 2020–2026

### 🎮 Playtime
- **MMORPGs and Sandboxes** have 10× the completionist hours of action games
- Longer games correlate weakly positively with Metacritic — depth is rewarded
- **Battle Royale**: technically endless — but "main story" estimate is 100h+ (seasonal)

---
*If this was useful, please upvote! 🙏*